# 02 Dodatkowe analizy – dobór słów (słownik obozów)

## 1. Konfiguracja

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.config import POLITICAL_SUBREDDITS, PROCESSED_DIR, RESULTS_DIR
from src.terms import find_mentions

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Terminy kandydujące (robocze – do zatwierdzenia, nie trafiają do `config.py`)

In [2]:
CANDIDATES = {
    "dem": {"party": ["democrat", "democrats", "democratic", "dem", "dems", "dnc"],
            "politician": ["obama", "biden", "clinton", "hillary", "kaine", "harris", "kamala", "walz"]},
    "rep": {"party": ["republican", "republicans", "gop", "rnc"],
            "politician": ["romney", "ryan", "trump", "pence", "vance"]},
}

## 3. Próba polityczna

In [3]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "subreddit", "camp", "cycle", "phase", "text"])
sample = sample[sample["subreddit"].isin(POLITICAL_SUBREDDITS)].reset_index(drop=True)
sample.shape

(455045, 6)

## 4. Liczba komentarzy z każdym terminem

In [4]:
rows = []
for camp, types in CANDIDATES.items():
    for term_type, terms in types.items():
        for term in terms:
            has_term = sample["text"].str.contains(rf"\b{term}\b", case=False, regex=True)
            counts = sample[has_term].groupby("cycle").size()
            rows.append({"camp": camp, "term_type": term_type, "term": term, **{cycle: counts.get(cycle, 0) for cycle in sorted(sample["cycle"].unique())}})
term_counts = pd.DataFrame(rows)
term_counts.to_csv(RESULTS_DIR / "camp_terms_counts.csv", index=False)
term_counts

,camp,term_type,term,2012,2016,2020,2024
0,dem,party,democrat,1324,2071,1988,1807
1,dem,party,democrats,2401,3796,4103,4005
2,dem,party,democratic,938,2405,1568,1222
3,dem,party,dem,188,411,519,503
4,dem,party,dems,483,876,1538,1638
5,dem,party,dnc,103,966,318,273
6,dem,politician,obama,7071,4511,2147,1149
7,dem,politician,biden,253,251,4292,6992
8,dem,politician,clinton,652,5139,809,469
9,dem,politician,hillary,152,6425,1084,520


## 5. Wzmianki dokładnie jednego obozu

In [5]:
mentions = find_mentions(sample["text"], CANDIDATES)
data = sample.join(mentions, how="inner")
data["target"] = (data["mentioned_camp"] == data["camp"]).map({True: "own", False: "out"})
coverage = data.groupby(["subreddit", "cycle", "phase", "target"]).size().unstack("target")
coverage.to_csv(RESULTS_DIR / "camp_terms_coverage.csv")
coverage

target                             out   own
subreddit    cycle phase                    
Conservative 2012  bazowa          294   129
                   powyborcza      252    67
                   przedwyborcza  1303   909
                   przejsciowa     247   128
                   szczytowa       312   202
...                                ...   ...
democrats    2024  bazowa          960   400
                   powyborcza      289   218
                   przedwyborcza  2628  1508
                   przejsciowa     495   289
                   szczytowa       488   336

[80 rows x 2 columns]

In [6]:
print(f"komentarzy z dokładnie jednym obozem: {len(data)} z {len(sample)} ({len(data) / len(sample):.1%})")

komentarzy z dokładnie jednym obozem: 90478 z 455045 (19.9%)
